[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kudp27-ops/Quantpath/blob/main/college-admissions-calculator/CollegeAdmissions.ipynb)

# College Admissions Calculator

Two tools in one notebook, inspired by the kind of admissions-consulting products
former admissions deans (e.g. John Morganelli's "BluePrint" at Ivy Tutors Network) sell:

1. **Chance Calculator** — estimates admission probability at a list of colleges from
   a student's academic profile (GPA, test scores, course rigor, extracurricular
   strength, and "hooks" like legacy/first-gen/recruited athlete), and sorts schools
   into Reach / Target / Likely / Safety.
2. **BluePrint-style Report Generator** — takes an intake questionnaire (interests,
   background, goals, grade level) and produces a personalized narrative report:
   an application "theme", suggested majors, recommended activities, independent
   project ideas, summer program ideas, essay brainstorm prompts, a financial-aid
   overview, and a year-by-year timeline.

**Disclaimer:** The college stats below (acceptance rate, SAT/ACT/GPA ranges) are
rough, illustrative approximations of recently published Common Data Set figures —
not live data. Refresh `COLLEGE_DB` with current numbers before relying on this for
real decisions. This tool gives a statistical estimate, not a guarantee; it does not
use race as a factor, consistent with current U.S. admissions law.

In [ ]:
import math
import textwrap

ACT_TO_SAT = {
    36: 1590, 35: 1560, 34: 1530, 33: 1500, 32: 1470, 31: 1440, 30: 1420,
    29: 1390, 28: 1360, 27: 1330, 26: 1300, 25: 1270, 24: 1240, 23: 1210,
    22: 1180, 21: 1150, 20: 1110, 19: 1070, 18: 1030, 17: 990, 16: 950,
    15: 910, 14: 860, 13: 800, 12: 760, 11: 710, 10: 670,
}

HOOK_BONUS_LOGIT = {
    "legacy": 0.4,
    "first_gen": 0.25,
    "recruited_athlete": 1.8,
    "geographic_diversity": 0.2,
    "demonstrated_interest": 0.1,
}

ACADEMIC_INDEX_WEIGHT_TEST = 0.6
ACADEMIC_INDEX_WEIGHT_GPA = 0.4
SENSITIVITY_K = 1.3
GPA_STD = 0.25

In [ ]:
# Illustrative, approximate Common Data Set-style figures. Replace with current
# data for real use. acceptance_rate is overall (not need-blind-adjusted).
COLLEGE_DB = [
    {"name": "Harvard University", "acceptance_rate": 0.034, "sat_25": 1500, "sat_75": 1580, "act_25": 34, "act_75": 36, "avg_gpa": 4.00, "need_blind": True, "pct_need_met": 100},
    {"name": "Stanford University", "acceptance_rate": 0.037, "sat_25": 1500, "sat_75": 1570, "act_25": 33, "act_75": 35, "avg_gpa": 4.00, "need_blind": True, "pct_need_met": 100},
    {"name": "MIT", "acceptance_rate": 0.040, "sat_25": 1520, "sat_75": 1580, "act_25": 35, "act_75": 36, "avg_gpa": 4.00, "need_blind": True, "pct_need_met": 100},
    {"name": "Princeton University", "acceptance_rate": 0.039, "sat_25": 1500, "sat_75": 1580, "act_25": 34, "act_75": 36, "avg_gpa": 3.95, "need_blind": True, "pct_need_met": 100},
    {"name": "Yale University", "acceptance_rate": 0.046, "sat_25": 1500, "sat_75": 1580, "act_25": 33, "act_75": 35, "avg_gpa": 4.00, "need_blind": True, "pct_need_met": 100},
    {"name": "Columbia University", "acceptance_rate": 0.039, "sat_25": 1500, "sat_75": 1570, "act_25": 34, "act_75": 35, "avg_gpa": 3.95, "need_blind": True, "pct_need_met": 100},
    {"name": "University of Pennsylvania", "acceptance_rate": 0.059, "sat_25": 1500, "sat_75": 1570, "act_25": 34, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 100},
    {"name": "Cornell University", "acceptance_rate": 0.075, "sat_25": 1460, "sat_75": 1550, "act_25": 33, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 100},
    {"name": "Brown University", "acceptance_rate": 0.051, "sat_25": 1500, "sat_75": 1570, "act_25": 33, "act_75": 35, "avg_gpa": 3.95, "need_blind": True, "pct_need_met": 100},
    {"name": "Dartmouth College", "acceptance_rate": 0.062, "sat_25": 1480, "sat_75": 1560, "act_25": 32, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 100},
    {"name": "Duke University", "acceptance_rate": 0.060, "sat_25": 1490, "sat_75": 1570, "act_25": 34, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 100},
    {"name": "Northwestern University", "acceptance_rate": 0.072, "sat_25": 1470, "sat_75": 1550, "act_25": 33, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 100},
    {"name": "Johns Hopkins University", "acceptance_rate": 0.070, "sat_25": 1500, "sat_75": 1560, "act_25": 34, "act_75": 35, "avg_gpa": 3.92, "need_blind": True, "pct_need_met": 100},
    {"name": "University of Chicago", "acceptance_rate": 0.054, "sat_25": 1500, "sat_75": 1570, "act_25": 34, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 100},
    {"name": "Vanderbilt University", "acceptance_rate": 0.056, "sat_25": 1500, "sat_75": 1570, "act_25": 34, "act_75": 35, "avg_gpa": 3.85, "need_blind": True, "pct_need_met": 100},
    {"name": "Rice University", "acceptance_rate": 0.089, "sat_25": 1500, "sat_75": 1570, "act_25": 33, "act_75": 35, "avg_gpa": 3.95, "need_blind": True, "pct_need_met": 100},
    {"name": "University of Notre Dame", "acceptance_rate": 0.129, "sat_25": 1440, "sat_75": 1530, "act_25": 33, "act_75": 35, "avg_gpa": 3.90, "need_blind": True, "pct_need_met": 95},
    {"name": "Georgetown University", "acceptance_rate": 0.120, "sat_25": 1440, "sat_75": 1540, "act_25": 32, "act_75": 34, "avg_gpa": 3.90, "need_blind": False, "pct_need_met": 90},
    {"name": "University of Michigan", "acceptance_rate": 0.177, "sat_25": 1350, "sat_75": 1530, "act_25": 31, "act_75": 34, "avg_gpa": 3.85, "need_blind": False, "pct_need_met": 85},
    {"name": "University of Virginia", "acceptance_rate": 0.187, "sat_25": 1370, "sat_75": 1530, "act_25": 31, "act_75": 34, "avg_gpa": 3.95, "need_blind": False, "pct_need_met": 80},
    {"name": "UC Berkeley", "acceptance_rate": 0.116, "sat_25": 1330, "sat_75": 1530, "act_25": 30, "act_75": 35, "avg_gpa": 3.90, "need_blind": False, "pct_need_met": 75},
    {"name": "UCLA", "acceptance_rate": 0.086, "sat_25": 1310, "sat_75": 1530, "act_25": 29, "act_75": 34, "avg_gpa": 3.95, "need_blind": False, "pct_need_met": 75},
    {"name": "New York University", "acceptance_rate": 0.128, "sat_25": 1430, "sat_75": 1550, "act_25": 32, "act_75": 34, "avg_gpa": 3.70, "need_blind": False, "pct_need_met": 70},
    {"name": "Boston University", "acceptance_rate": 0.140, "sat_25": 1410, "sat_75": 1520, "act_25": 31, "act_75": 34, "avg_gpa": 3.70, "need_blind": False, "pct_need_met": 70},
    {"name": "The Ohio State University", "acceptance_rate": 0.530, "sat_25": 1240, "sat_75": 1450, "act_25": 27, "act_75": 32, "avg_gpa": 3.70, "need_blind": False, "pct_need_met": 60},
    {"name": "Penn State University", "acceptance_rate": 0.550, "sat_25": 1180, "sat_75": 1390, "act_25": 25, "act_75": 31, "avg_gpa": 3.60, "need_blind": False, "pct_need_met": 55},
    {"name": "Arizona State University", "acceptance_rate": 0.900, "sat_25": 1110, "sat_75": 1370, "act_25": 21, "act_75": 29, "avg_gpa": 3.50, "need_blind": False, "pct_need_met": 50},
    {"name": "University of Alabama", "acceptance_rate": 0.800, "sat_25": 1100, "sat_75": 1340, "act_25": 23, "act_75": 31, "avg_gpa": 3.60, "need_blind": False, "pct_need_met": 45},
]

In [ ]:
def act_to_sat(act_score):
    act_score = max(min(round(act_score), 36), 10)
    return ACT_TO_SAT[act_score]


def percentile_range_to_zscore(value, p25, p75):
    mean = (p25 + p75) / 2
    std = (p75 - p25) / 1.349  # distance between the 25th/75th percentiles of a normal curve
    return (value - mean) / std


def academic_index_z(student, school):
    sat_equivalent = student.get("sat") or act_to_sat(student["act"])
    test_z = percentile_range_to_zscore(sat_equivalent, school["sat_25"], school["sat_75"])
    gpa_z = (student["gpa"] - school["avg_gpa"]) / GPA_STD

    rigor_adj = (student.get("course_rigor", 7) - 7) * 0.1
    ec_adj = (student.get("ec_strength", 5) - 5) * 0.15

    return (
        ACADEMIC_INDEX_WEIGHT_TEST * test_z
        + ACADEMIC_INDEX_WEIGHT_GPA * gpa_z
        + rigor_adj
        + ec_adj
    )


def hook_bonus(student):
    return sum(HOOK_BONUS_LOGIT.get(hook, 0) for hook in student.get("hooks", []))


def estimate_admission_probability(student, school):
    base_rate = min(max(school["acceptance_rate"], 0.001), 0.999)
    base_logit = math.log(base_rate / (1 - base_rate))

    z = academic_index_z(student, school)
    logit = base_logit + SENSITIVITY_K * z + hook_bonus(student)

    probability = 1 / (1 + math.exp(-logit))
    return min(max(probability, 0.01), 0.98)


def classify_tier(probability):
    if probability < 0.15:
        return "Reach"
    if probability < 0.40:
        return "Target"
    if probability < 0.70:
        return "Likely"
    return "Safety"

In [ ]:
def run_chances(student, college_db=COLLEGE_DB):
    rows = []
    for school in college_db:
        p = estimate_admission_probability(student, school)
        rows.append({
            "school": school["name"],
            "estimated_chance": p,
            "tier": classify_tier(p),
            "school_accept_rate": school["acceptance_rate"],
        })
    rows.sort(key=lambda r: r["estimated_chance"])
    return rows


def print_chances(rows):
    print(f"{'School':30} {'Est. Chance':>12} {'Tier':>8} {'Base Rate':>10}")
    print("-" * 64)
    for r in rows:
        print(
            f"{r['school']:30} {r['estimated_chance']*100:>11.1f}% "
            f"{r['tier']:>8} {r['school_accept_rate']*100:>9.1f}%"
        )

## BluePrint-style Report Generator

Mirrors the structure of Morganelli's BluePrint product: an intake questionnaire
feeds a curated suggestion bank keyed by the student's primary interest area, plus
a generic but personalized timeline and financial-aid overview.

In [ ]:
INTEREST_BANK = {
    "stem_engineering": {
        "majors": ["Mechanical Engineering", "Electrical Engineering", "Materials Science"],
        "activities": ["FIRST Robotics or a robotics club", "Science Olympiad", "Math team"],
        "project_ideas": [
            "Design and document a robotics or hardware project with a public writeup",
            "Enter a regional science fair or research competition",
        ],
        "summer_programs": ["RSI", "MIT MITES/MOSTEC", "a university engineering pre-college program"],
        "essay_prompts": [
            "A time you took apart, fixed, or built something and what it taught you",
            "A problem in your community you'd want to solve with engineering",
        ],
    },
    "computer_science": {
        "majors": ["Computer Science", "Data Science", "Computer Engineering"],
        "activities": ["Coding club or competitive programming (USACO)", "Hackathons", "Open-source contributions"],
        "project_ideas": [
            "Ship a real app or tool used by other students and document the build",
            "Contribute to an open-source project and track your pull requests",
        ],
        "summer_programs": ["a CS research REU", "Jane Street/AlphaSignal-style coding camp", "a university CS pre-college program"],
        "essay_prompts": [
            "A bug or failure that taught you more than a success did",
            "Something you built because the tool you needed didn't exist",
        ],
    },
    "business_economics": {
        "majors": ["Economics", "Business Administration", "Finance"],
        "activities": ["DECA or FBLA", "Investing/finance club", "A small student-run business"],
        "project_ideas": [
            "Start a small business or freelance service and track real numbers",
            "Run an independent research project analyzing a local market or industry",
        ],
        "summer_programs": ["a pre-college business program", "an investing or trading competition", "a local internship at a small business"],
        "essay_prompts": [
            "A decision involving money or risk you got wrong, and what changed after",
            "A local economic problem you noticed and how you'd fix it",
        ],
    },
    "biology_premed": {
        "majors": ["Biology", "Neuroscience", "Public Health"],
        "activities": ["HOSA", "Hospital or clinic volunteering", "Lab research with a local university"],
        "project_ideas": [
            "Independent research project with a faculty mentor, written up formally",
            "A public health awareness campaign for your school or town",
        ],
        "summer_programs": ["a university pre-med/research program", "a hospital shadowing program", "an REU in biology"],
        "essay_prompts": [
            "A moment that shaped how you think about health, illness, or care",
            "A scientific question you can't stop thinking about",
        ],
    },
    "humanities_english": {
        "majors": ["English", "Comparative Literature", "Philosophy"],
        "activities": ["Literary magazine or school newspaper", "Debate team", "Model UN"],
        "project_ideas": [
            "Self-publish a collection of writing (blog, zine, or chapbook)",
            "Run an independent research project on a text or author you love",
        ],
        "summer_programs": ["a creative writing pre-college program", "a journalism workshop", "a humanities research program"],
        "essay_prompts": [
            "A book or idea that changed how you see the world",
            "A piece of writing you're proud of and why it mattered to you",
        ],
    },
    "social_sciences_government": {
        "majors": ["Political Science", "International Relations", "Sociology"],
        "activities": ["Model UN", "Student government", "Local political campaign volunteering"],
        "project_ideas": [
            "An independent policy research paper on a local or national issue",
            "Organize a civic engagement event (voter registration, town hall, etc.)",
        ],
        "summer_programs": ["a pre-college government/policy program", "a congressional internship", "a Model UN summer institute"],
        "essay_prompts": [
            "A local issue you'd want to change and what you've already done about it",
            "A moment you disagreed with authority and how you handled it",
        ],
    },
    "arts_design": {
        "majors": ["Studio Art", "Architecture", "Graphic/Industrial Design"],
        "activities": ["Art club or portfolio class", "Theater or stage design", "Freelance design work"],
        "project_ideas": [
            "Build a public portfolio website showcasing a cohesive body of work",
            "A community mural, exhibit, or design commission",
        ],
        "summer_programs": ["a pre-college art/design program (RISD, Parsons, etc.)", "a portfolio-building workshop"],
        "essay_prompts": [
            "A piece you made that didn't turn out as planned, and what you learned",
            "The first time you realized you wanted to make things for a living",
        ],
    },
    "undecided": {
        "majors": ["Undeclared / Liberal Arts", "Interdisciplinary Studies"],
        "activities": ["Two or three clubs explored deeply rather than many shallowly", "A job or internship outside school"],
        "project_ideas": ["An independent project that combines two unrelated interests"],
        "summer_programs": ["a broad pre-college exploratory program", "a local internship"],
        "essay_prompts": ["A time your interests collided in an unexpected way"],
    },
}

TIMELINE = {
    9: [
        "Take the most rigorous courses you can handle; grades start counting now.",
        "Try 2-3 extracurriculars to find what sticks; depth beats breadth later.",
        "Start a simple resume/activity log so you don't forget details later.",
    ],
    10: [
        "Take the PSAT; identify a baseline SAT/ACT score and a study plan.",
        "Narrow extracurriculars to 1-3 you can go deep on through senior year.",
        "Start an independent project tied to your main interest area.",
        "Begin a casual college list (20-30 schools) based on fit, not just rank.",
    ],
    11: [
        "Take the SAT/ACT (and retake if needed); aim to finish testing by spring.",
        "Take 2+ AP/IB/honors courses in your strongest subjects.",
        "Deepen your signature project or activity into something show-able.",
        "Request letters of recommendation from junior-year teachers in the spring.",
        "Visit campuses or attend virtual info sessions; narrow your list to 8-15 schools.",
        "Draft your Common App personal statement over the summer.",
    ],
    12: [
        "Finalize your college list across reach/target/likely/safety tiers.",
        "Complete the Common App, supplements, and essays well before deadlines.",
        "File the FAFSA (opens Oct 1) and CSS Profile if required by your schools.",
        "Submit early action/decision applications by November, regular by January.",
        "Compare financial aid offers in the spring before committing by May 1.",
    ],
}

In [ ]:
def generate_blueprint_report(student):
    bank = INTEREST_BANK.get(student.get("interest_area", "undecided"), INTEREST_BANK["undecided"])
    grade = student.get("grade_level", 11)
    name = student.get("name", "Student")

    lines = []
    lines.append(f"BLUEPRINT REPORT — {name} (Grade {grade})")
    lines.append("=" * 60)

    lines.append("\nAPPLICATION THEME")
    theme = (
        f"{name}'s application centers on {student.get('interest_area', 'an evolving interest').replace('_', ' ')}, "
        f"grounded in: \"{student.get('background_story', 'a personal story still being shaped').strip()}\" "
        f"and aimed at: \"{student.get('goals', 'a goal still being defined').strip()}\"."
    )
    lines.append(textwrap.fill(theme, width=78))

    lines.append("\nSUGGESTED MAJORS")
    lines.extend(f"  - {m}" for m in bank["majors"])

    lines.append("\nRECOMMENDED ACTIVITIES")
    lines.extend(f"  - {a}" for a in bank["activities"])

    lines.append("\nINDEPENDENT PROJECT IDEAS")
    lines.extend(f"  - {p}" for p in bank["project_ideas"])

    lines.append("\nSUMMER PROGRAM IDEAS")
    lines.extend(f"  - {s}" for s in bank["summer_programs"])

    lines.append("\nCOLLEGE ESSAY BRAINSTORM PROMPTS")
    lines.extend(f"  - {e}" for e in bank["essay_prompts"])

    lines.append("\nFINANCIAL AID OVERVIEW")
    lines.append(textwrap.fill(
        "File the FAFSA as soon as it opens (Oct 1) and the CSS Profile for any "
        "school that requires it. Need-blind schools admit without considering "
        "ability to pay; 'meets full need' schools cover the gap between aid and "
        "cost. Always run each school's net price calculator before assuming cost "
        "based on the sticker price.", width=78
    ))

    target_schools = student.get("target_schools", [])
    if target_schools:
        lines.append("\n  Aid posture of your target schools:")
        by_name = {s["name"]: s for s in COLLEGE_DB}
        for name_ in target_schools:
            s = by_name.get(name_)
            if s:
                posture = "need-blind" if s["need_blind"] else "need-aware"
                lines.append(f"    - {name_}: {posture}, meets ~{s['pct_need_met']}% of demonstrated need")

    lines.append("\nTIMELINE")
    for g in range(grade, 13):
        lines.append(f"  Grade {g}:")
        lines.extend(f"    - {item}" for item in TIMELINE.get(g, []))

    return "\n".join(lines)

## Example

Fill in a student profile and run both tools.

In [ ]:
sample_student = {
    "name": "Jordan",
    "grade_level": 11,
    "gpa": 3.85,                 # unweighted, 4.0 scale
    "sat": 1430,                  # or use "act": 32 instead
    "course_rigor": 8,             # 1-10, 7 = typical college-prep rigor
    "ec_strength": 7,              # 1-10, 5 = solid but unremarkable
    "hooks": ["first_gen"],         # any of HOOK_BONUS_LOGIT keys
    "interest_area": "computer_science",
    "background_story": "Grew up fixing family members' computers and got hooked on how things work under the hood.",
    "goals": "Build software that makes technical tools accessible to people without a CS background.",
    "target_schools": ["Cornell University", "University of Michigan", "Boston University"],
}

chances = run_chances(sample_student)
print_chances(chances)

print("\n" + "=" * 64 + "\n")

print(generate_blueprint_report(sample_student))